Hasta ahora se han unificado los dos dominios, se han chunkeado y ahora se van a hacer un embedding.

In [1]:
# instalar librerías

!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.7 MB/s eta 0:00:00


In [2]:
# cargar chunks

import json
import os

RUTA_CHUNKS = "corpus_aml_chunks.json"

with open(RUTA_CHUNKS, "r", encoding="utf-8") as f:
    chunks = json.load(f)

chunks_filtrados = [
    c for c in chunks
]

print("Chunks originales:", len(chunks))
print("Chunks filtrados:", len(chunks_filtrados))

Chunks originales: 1529
Chunks filtrados: 1529


In [3]:
# generar embeddings

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = "intfloat/multilingual-e5-base"

model = SentenceTransformer(modelo_embeddings)

textos = [
    "passage: " + c["texto_chunk"]
    for c in chunks_filtrados
]

embeddings = model.encode(
    textos,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

print("Shape embeddings:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Shape embeddings: (1529, 768)


In [4]:
# crear índice FAISS

import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vectores en el índice:", index.ntotal)

Vectores en el índice: 1529


In [5]:
# guardar índice y metadatos

CARPETA_EMBEDDINGS = "corpus_rag_embeddings"
os.makedirs(CARPETA_EMBEDDINGS, exist_ok=True)

faiss.write_index(index, f"{CARPETA_EMBEDDINGS}/indice_faiss_aml.index")

with open(f"{CARPETA_EMBEDDINGS}/chunks_filtrados.json", "w", encoding="utf-8") as f:
    json.dump(chunks_filtrados, f, ensure_ascii=False, indent=2)

print("Índice FAISS y chunks guardados.")

Índice FAISS y chunks guardados.


In [6]:
# dominio ya vectorizado e indexado

import shutil
from google.colab import files

shutil.make_archive("corpus_rag_embeddings", "zip", "corpus_rag_embeddings")
files.download("corpus_rag_embeddings.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# función de búsqueda de prueba

def buscar(query, top_k=5):
    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    resultados = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks_filtrados[idx].copy()
        chunk["score"] = float(score)
        resultados.append(chunk)

    return resultados

In [8]:
# primera consulta de prueba

resultados = buscar("¿Qué obligaciones tienen las entidades financieras ante una persona sancionada?", top_k=5)

for r in resultados:
    print("=" * 80)
    print("Score:", r["score"])
    print("Chunk:", r["chunk_id"])
    print("Documento:", r["doc_id"], "-", r["titulo"])
    print("Fuente:", r["fuente"])
    print("Categoría:", r["categoria"])
    print()
    print(r["texto_chunk"][:1200])
    print()

Score: 0.8480923175811768
Chunk: 106_0028
Documento: 106 - Directiva UE 2024/1640 relativa a los mecanismos que deben establecer los Estados miembros a efectos de la prevención de la utilización del sistema financiero para el blanqueo de capitales o la financiación del terrorismo
Fuente: BOE
Categoría: legislacion_bcft

sanciones pecuniarias y la aplicación de medidas administrativas, y la imposición de sanciones penales, con arreglo al Derecho nacional, no vulnere el principio non bis in idem. (103) En el caso de las entidades obligadas que sean personas jurídicas, las infracciones relativas a los requisitos de LBC/LFT se producen tras la actuación, o bajo la responsabilidad, de las personas físicas facultadas para dirigir sus actividades, en particular a través de agentes, distribuidores u otras personas que actúen en nombre de la entidad obligada. A fin de garantizar que las actuaciones de supervisión en respuesta a tales infracciones sean eficaces, la entidad obligada también debe 

In [9]:
resultados = buscar("¿Qué debe hacerse si se detecta una coincidencia con una persona o entidad incluida en una lista de sanciones financieras?", top_k=5)

for r in resultados:
    print("=" * 80)
    print("Score:", r["score"])
    print("Chunk:", r["chunk_id"])
    print("Documento:", r["doc_id"], "-", r["titulo"])
    print("Fuente:", r["fuente"])
    print("Categoría:", r["categoria"])
    print()
    print(r["texto_chunk"][:1200])
    print()

Score: 0.8483837842941284
Chunk: 080_0006
Documento: 080 - Buenas prácticas Aplicación de listas de personas y entidades sujetas a sanciones y contramedidas financieras internacionales
Fuente: SEPBLAC
Categoría: procedimiento

Los procedimientos del sistema de alertas pueden prever que no se generen nuevas alertas sobre personas concretas respecto de las que ya se ha comprobado con certeza que no son las personas incluidas en listas. Ello puede reducir de forma apreciable la carga de trabajo asociada a la gestión de alertas. 8 [Página 11] BP 24 Aplicar medidas de diligencia debida reforzada a las personas sobre las que no se puede afirmar con certeza que no están incluidas en listas En algunos casos puede ser difícil asegurar que la persona analizada no está incluida en listas. Puede pensarse, por ejemplo, en los casos de homonimia, en los que, si bien puede concluirse con razonable seguridad que se trata de ese caso y que la persona no está incluida en listas, se carece de información